In [1]:
#!/g/data/xp65/public/apps/med_conda_scripts/analysis3-25.07.d/bin/python3
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

from dask.distributed import Client, wait
import os, sys
import psutil

sys.path.append('/home/548/cd3022/repos/Irradiance-comparisons/Irradiance-comparisons')
import logger
from read_datasets import read_dataset

LOG = logger.get_logger(__name__)
year = 2017
# qsub -I -q normal -P er8 -l walltime=2:00:00,ncpus=24,mem=120GB,jobfs=100MB,storage=gdata/xp65+gdata/er8+gdata/ob53+gdata/rt52+gdata/rv74+gdata/su28

# Set up Dask

In [2]:
client = Client(
    n_workers=48,
    threads_per_worker=1
)
client

2025-10-02 15:46:50,236:py.warnings:WARNING: /g/data/xp65/public/apps/med_conda/envs/analysis3-25.06/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45899 instead
  warnings.warn(

2025-10-02 15:46:50,236:py.warnings:WARNING: /g/data/xp65/public/apps/med_conda/envs/analysis3-25.06/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45899 instead
  warnings.warn(



Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/45899/status,
Dashboard: /proxy/45899/status,Workers: 48
Total threads: 48,Total memory: 95.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34583,Workers: 0
Dashboard: /proxy/45899/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:39047,Total threads: 1
Dashboard: /proxy/44141/status,Memory: 1.98 GiB
Nanny: tcp://127.0.0.1:38451,


2025-10-02 15:50:58,953 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 786025758db5a31d79bec6b57ae15407 initialized by task ('rechunk-merge-rechunk-split-rechunk-transfer-012a03e0e36befe4b412ded6fe6f8659', 96, 1, 2, 96, 1, 3) executed on worker tcp://127.0.0.1:39047
2025-10-02 15:50:59,152 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 0d875a10eea2f486790fddb92f5bdc91 initialized by task ('rechunk-merge-rechunk-split-rechunk-transfer-012a03e0e36befe4b412ded6fe6f8659', 96, 0, 2, 96, 0, 3) executed on worker tcp://127.0.0.1:46439
2025-10-02 15:51:00,106 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 2268f1b3aa8722f13d70358ee9685f4f initialized by task ('rechunk-merge-rechunk-split-rechunk-transfer-012a03e0e36befe4b412ded6fe6f8659', 96, 0, 4, 96, 0, 6) executed on worker tcp://127.0.0.1:43689
2025-10-02 15:51:00,204 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle a3cbbc11f199e631b48c76bbdb3741a0 initialized by task ('rechunk-merge-rechunk

# Prepare Data

In [3]:
BARRA = Path('/g/data/ob53/BARRA2/output/reanalysis/')
BARRA_R2 = BARRA / "AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1"
BARRA_R2_DIR = BARRA_R2 / "1hr"
var = 'rsds'


files = sorted([f for f in BARRA_R2_DIR.glob(f'{var}/latest/*1hr_20*.nc')])

def _preprocess(ds):
    return ds.sel(lat=slice(-44.5, -10), lon=slice(112, 156.26))
barra_r2 = xr.open_mfdataset(
    files,
    chunks={'time':24, 'lat':256, 'lon':256},
    concat_dim='time',
    combine='nested',
    data_vars='minimal',
    coords='minimal',
    compat='override',
    parallel=True,
    preprocess=_preprocess
)

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.06/lib/python3.11/site-packages/dask/_task_spec.py:763: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 24. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.06/lib/python3.11/site-packages/dask/_task_spec.py:763: UserWarning: The specified chunks separate the stored chunks along dimension "lat" starting at index 256. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.06/lib/python3.11/site-packages/dask/_task_spec.py:763: UserWarning: The specified chunks separate the stored chunks along dimension "lon" starting at index 256. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs

In [4]:
ds_ct = xr.open_zarr("/g/data/su28/himawari-ahi/cloud/ct/aus_regional_domain/S_NWC_CT_HIMA08_HIMA-N-NR.zarr/")

lat=slice(-10, -44.5)
lon=slice(112, 156.26)
ds_ct = ds_ct.sel(
    lat=lat,
    lon=lon,
)

In [5]:
himawari_list = []
for month in range (1, 13):
    date = f'{year}-{month:02d}' 
    himawari = read_dataset(
            dataset='himawari',
            resolution='hourly',
            date=date
        )
    himawari_list.append(himawari)
him_ds = xr.concat(himawari_list, dim='time', data_vars='minimal')

# Process

In [6]:
# time slice based on himawari
start = him_ds.isel(time=0).time
end = him_ds.isel(time=-1).time
barra_date = barra_r2.sel(
    time=slice(start, end)
)

# NEW REGRIDDING METHOD
him_ds = him_ds.interp(
    lat=barra_date.lat,
    lon=barra_date.lon,
    method='linear'
)
diff = him_ds.ghi - barra_date.rsds

ds_ct_date = ds_ct.sel(
    time=diff.time,
    method='nearest'
)

ds_ct_date = ds_ct_date.interp(
    lat=diff.lat,
    lon=diff.lon,
    method='nearest'
)

if diff["time"].to_index().duplicated().any():
    diff = diff.sel(time=~diff.get_index("time").duplicated())

if ds_ct_date["time"].to_index().duplicated().any():
    ds_ct_date = ds_ct_date.sel(time=~ds_ct_date.get_index("time").duplicated())

final_ds = xr.Dataset(
    {
        "ghi_diff": diff,
        "ct": ds_ct_date.ct
    }
)

In [7]:
# Remove any pre-existing chunk encodings
for v in final_ds:
    if "chunks" in final_ds[v].encoding:
        del final_ds[v].encoding["chunks"]

final_ds = final_ds.chunk({"time": 24, "lat": 157, "lon": 31})
# ensure all variables match these chunks


file_path = Path("/scratch/er8/cd3022/Irradiance-comparisons/")
os.makedirs(file_path, exist_ok=True)
file_name = f'himawari_barrar2_diffct_{year}'

final_ds.to_zarr(f"{file_path}/{file_name}.zarr", mode="w", consolidated=True, zarr_format=2)

2025-10-02 15:49:25,827:py.warnings:WARNING: /g/data/xp65/public/apps/med_conda/envs/analysis3-25.06/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 44.39 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(

2025-10-02 15:49:25,827:py.warnings:WARNING: /g/data/xp65/public/apps/med_conda/envs/analysis3-25.06/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 44.39 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(

This may